In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run ../../../utils/reader_utils

In [0]:
%run ../../../utils/writer_utils

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today()

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False)
START_DATE = configs.get("start_date") or today - timedelta(days=1)
END_DATE = configs.get("end_date") or today + timedelta(days=1)
CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
# Constants

SOURCE_TABLE_CONF = {
    "table": "vehicles_raw",
    "schema": "bronze",
    "dedupe_desc": ["vehicle_id"],
    "order_col": "last_updated",
    "timestamp_col": "_ingested_at"
}

TARGET_TABLE_CONF = {
    "table": "pyspark_scd1_vehicles",
    "schema": "silver",
    "merge_keys": ["vehicle_id"],
    "primary_key": "vehicle_id"
}

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
vehicles_df = read_delta_table(SOURCE_TABLE_CONF, START_DATE, END_DATE)

In [0]:
final_vehicles_df = (vehicles_df.filter(F.col("vehicle_id").isNotNull())
        .withColumn('capacity_kg', F.col("capacity_kg").cast(IntegerType()))
        .withColumn('cold_chain_capable', F.col("cold_chain_capable").cast(BooleanType()))
        .withColumn('commissioned_date', F.col("commissioned_date").cast(DateType()))
        .withColumn('last_updated', F.to_timestamp(F.col("last_updated")))
        .withColumn('_insert_update_ts', F.current_timestamp().cast(TimestampType()))
        .select("capacity_kg",
                "cold_chain_capable",
                "commissioned_date",
                "home_depot",
                "last_updated",
                "model",
                "plate_number",
                "vehicle_id",
                "vehicle_type",
                "_insert_update_ts"))

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    target_table = f"{TARGET_TABLE_CONF.get('schema')}.{TARGET_TABLE_CONF.get('table')}"
    primary_key = TARGET_TABLE_CONF.get('primary_key')

    (final_vehicles_df
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())
    
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY (vehicle_id, _insert_update_ts)")
    spark.sql(f"ALTER TABLE {target_table} ALTER COLUMN {primary_key} SET NOT NULL")
    spark.sql(f"ALTER TABLE {target_table} ADD CONSTRAINT {primary_key}_pk PRIMARY KEY ({primary_key})")
    
    dbutils.notebook.exit(f"Initial Table: {target_table} Setup Finished Successfully")

In [0]:
upsert_to_delta(final_vehicles_df, TARGET_TABLE_CONF)